In [1]:
# ============================================
# CELL 1: Install packages
# ============================================
!pip install faster-whisper
!pip install fastapi
!pip install uvicorn
!pip install python-multipart
!pip install pyngrok
!pip install nest-asyncio
!pip install indic-transliteration
!pip install openai
!pip install httpx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 7.8 MB/s eta 0:00:00


In [2]:
# ============================================
# CELL 2: Load Whisper Model
# ============================================
from faster_whisper import WhisperModel

print("📥 Downloading Whisper Large V3 model...")
print("⏳ This will take 3-5 minutes first time only...")

model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

print("✅ Model loaded successfully!")
print("🎯 Ready to transcribe!")

📥 Downloading Whisper Large V3 model...
⏳ This will take 3-5 minutes first time only...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Model loaded successfully!
🎯 Ready to transcribe!


In [3]:
# ============================================
# CELL 3: Solid Whisper Prompt
# ============================================

WHISPER_INITIAL_PROMPT = """
You are transcribing speech that can be English, Romanized Nepali, or Nepglish (mix).

YOUR JOB: Output **only** clean Roman script exactly as spoken. Do NOT translate. Do NOT output Devanagari.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL OUTPUT LOGIC (for the whole system):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. ENGLISH SPEECH:
   - Romanized (your output) = Keep original English words
   - Devanagari (later)      = Phonetic Nepali script (e.g. "how are you" → हाई! हाउ आर यू?)

2. NEPALI SPEECH:
   - Romanized (your output) = Clean Romanized Nepali using standard spellings below
   - Devanagari (later)      = Proper Devanagari script (e.g. "timilai aaja kasto xa" → तिमीलाई आज कस्तो छ?)

3. NEPGLISH (MIX):
   - Romanized = Keep both English words and Romanized Nepali as spoken
   - Devanagari = Mixed conversion

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLES — WHAT YOU SHOULD OUTPUT:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ENGLISH SPEECH (output Romanized English):
spoken: "how are you"                    → output: how are you
spoken: "good morning how are you"       → output: good morning how are you
spoken: "i am tired today"               → output: i am tired today
spoken: "what are you doing"             → output: what are you doing
spoken: "thank you so much"              → output: thank you so much
spoken: "the meeting is cancelled"       → output: the meeting is cancelled
spoken: "see you tomorrow"               → output: see you tomorrow
spoken: "i found a bug"                  → output: i found a bug
spoken: "good night sleep well"          → output: good night sleep well
spoken: "the server is down"             → output: the server is down

NEPALI SPEECH (output clean Romanized Nepali):
spoken: "तिमीलाई आज कस्तो छ"             → output: timilai aaja kasto xa
spoken: "हे तिमी के गर्दैछौ"              → output: hey timi k gardai xau
spoken: "म ठिक छु"                        → output: ma thik xu
spoken: "मलाई थाहा छैन"                   → output: malai thaha xaina
spoken: "आज के भयो"                      → output: aaja k bhayo
spoken: "मेरो नाम कशिश हो"                → output: mero naam kashish ho
spoken: "मिटिङ क्यान्सेल भयो"             → output: meeting cancel bhayo
spoken: "म अफिसमा बिजी छु"                → output: ma office ma busy xu
spoken: "सर्भर डाउन छ"                    → output: server down xa
spoken: "धेरै स्ट्रेस्ड छु"               → output: dherai stressed xu
spoken: "अब काम सुरु गरौं"                → output: aba kaam suru garaam
spoken: "मैले इमेल गरेको"                 → output: maile email gareko
spoken: "तपाईको काम सकियो"                → output: tapaiko kaam sakiyo
spoken: "यो आइडिया राम्रो छ"              → output: yo idea ramro xa
spoken: "म धेरै थाकेको छु"                → output: ma dherai tired xu
spoken: "हामीसँग समय छैन"                 → output: hami sanga time xaina
spoken: "साँझमा भेटौला"                   → output: saanjh ma bhetaula
spoken: "एक छिन पर्खनु"                   → output: ek chin parkhanu
spoken: "फेरि भन्नुस् न"                   → output: feri bhannus na

NEPGLISH MIXED EXAMPLES:
spoken: "आज म धेरै tired छु"              → output: aaja ma dherai tired xu
spoken: "deadline आज छ"                   → output: deadline aaja xa
spoken: "meeting cancel भयो"              → output: meeting cancel bhayo
spoken: "म office मा busy छु"             → output: ma office ma busy xu
spoken: "finally done भयो"                → output: finally done bhayo

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STRICT ROMANIZED NEPALI SPELLING RULES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- छु   → xu
- छौ  → xau
- छ    → xa
- छन् → xan
- छैन → xaina
- गर्दै → gardai (never gardoyso, garde, shaudini, etc.)
- गर्छु → garxu
- हुन्छ → hunxa
- के    → ke or k
- तिमी  → timi (never time, timee, temi)
- हे    → hey
- भयो  → bhayo
- गरेको → gareko
- लाग्यो → lagyo
- थाहा  → thaha
- आज    → aaja
- धेरै  → dherai
- अहिले → aile
- अब    → aba

Never output Devanagari characters. Never translate. Only output clean Roman script following the examples above.
"""

print("✅ Solid Whisper prompt v3.0 loaded with clear English vs Nepali logic!")

✅ Solid Whisper prompt v3.0 loaded with clear English vs Nepali logic!


In [4]:
# ============================================
# CELL 4: Word & Sentence Maps
# ============================================

# ── PHRASE MAP (checked first, longest match wins) ──
PHRASE_MAP = {

    # ════════════════════════════════════════
    # PURE ENGLISH → Devanagari Phonetic
    # ════════════════════════════════════════

    # Greetings
    "hi how are you":                   "हाई! हाउ आर यू?",
    "hi how are you doing":             "हाई! हाउ आर यू डुइङ?",
    "hello how are you":                "हेलो! हाउ आर यू?",
    "hello how are you doing":          "हेलो! हाउ आर यू डुइङ?",
    "good morning how are you":         "गुड मर्निङ! हाउ आर यू?",
    "good morning how are you doing":   "गुड मर्निङ! हाउ आर यू डुइङ?",
    "good afternoon":                   "गुड आफ्टरनून!",
    "good evening":                     "गुड इभनिङ!",
    "good night":                       "गुड नाइट!",
    "good night sleep well":            "गुड नाइट! स्लिप वेल!",
    "hi there":                         "हाई देअर!",
    "hey there":                        "हे देअर!",
    "hey what is up":                   "हे! व्हाट इज अप?",
    "what is up":                       "व्हाट इज अप?",
    "how are you":                      "हाउ आर यू?",
    "how are you doing":                "हाउ आर यू डुइङ?",
    "how have you been":                "हाउ ह्याभ यू बिन?",
    "nice to meet you":                 "नाइस टु मिट यू!",
    "nice to meet you too":             "नाइस टु मिट यू टु!",
    "pleased to meet you":              "प्लिज्ड टु मिट यू!",
    "long time no see":                 "लङ टाइम नो सी!",
    "welcome":                          "वेलकम!",
    "welcome back":                     "वेलकम ब्याक!",

    # Responses
    "i am fine thank you":              "आइ एम फाइन, थ्याङ्क यू!",
    "i am fine thanks":                 "आइ एम फाइन, थ्याङ्क्स!",
    "i am doing well":                  "आइ एम डुइङ वेल!",
    "i am doing well thank you":        "आइ एम डुइङ वेल, थ्याङ्क यू!",
    "i am good":                        "आइ एम गुड!",
    "i am good thank you":              "आइ एम गुड, थ्याङ्क यू!",
    "not bad":                          "नट ब्याड!",
    "not bad thank you":                "नट ब्याड, थ्याङ्क यू!",
    "pretty good":                      "प्रिटी गुड!",
    "could be better":                  "कुड बी बेटर!",
    "same as always":                   "सेम एज अलवेज!",

    # Introductions
    "what is your name":                "व्हाट इज योर नेम?",
    "my name is":                       "माइ नेम इज",
    "where are you from":               "व्हेअर आर यू फ्रम?",
    "i am from nepal":                  "आइ एम फ्रम नेपाल!",
    "i am from kathmandu":              "आइ एम फ्रम काठमाडौँ!",
    "how old are you":                  "हाउ ओल्ड आर यू?",
    "what do you do":                   "व्हाट डु यू डु?",
    "where do you work":                "व्हेअर डु यू वर्क?",

    # Asking / Questions
    "what are you doing":               "व्हाट आर यू डुइङ?",
    "what are you doing right now":     "व्हाट आर यू डुइङ राइट नाउ?",
    "where are you going":              "व्हेअर आर यू गोइङ?",
    "where are you":                    "व्हेअर आर यू?",
    "what happened":                    "व्हाट ह्यापेन्ड?",
    "what is going on":                 "व्हाट इज गोइङ अन?",
    "what do you think":                "व्हाट डु यू थिङ्क?",
    "what do you need":                 "व्हाट डु यू नीड?",
    "what do you want":                 "व्हाट डु यू वान्ट?",
    "can you help me":                  "क्यान यू हेल्प मी?",
    "can you hear me":                  "क्यान यू हिअर मी?",
    "do you understand":                "डु यू अन्डरस्ट्यान्ड?",
    "did you get it":                   "डिड यू गेट इट?",
    "are you sure":                     "आर यू स्योर?",
    "are you ready":                    "आर यू रेडी?",
    "are you okay":                     "आर यू ओके?",
    "is everything okay":               "इज एभ्रीथिङ ओके?",
    "is that okay":                     "इज द्याट ओके?",
    "what time is it":                  "व्हाट टाइम इज इट?",
    "what is the plan":                 "व्हाट इज द प्लान?",
    "what is the update":               "व्हाट इज द अपडेट?",
    "what is the status":               "व्हाट इज द स्टाटस?",
    "when is the deadline":             "व्हेन इज द डेडलाइन?",
    "how long will it take":            "हाउ लङ विल इट टेक?",
    "how much does it cost":            "हाउ मच डज इट कस्ट?",
    "who is responsible":               "हू इज रिस्पन्सिबल?",
    "which one is better":              "व्हिच वन इज बेटर?",
    "why is this happening":            "व्हाई इज दिस ह्यापेनिङ?",

    # Common statements
    "i am tired":                       "आइ एम टायर्ड!",
    "i am tired today":                 "आइ एम टायर्ड टुडे!",
    "i am very tired":                  "आइ एम भेरी टायर्ड!",
    "i am so tired":                    "आइ एम सो टायर्ड!",
    "i am exhausted":                   "आइ एम एग्जस्टेड!",
    "i am busy":                        "आइ एम बिजी!",
    "i am busy right now":              "आइ एम बिजी राइट नाउ!",
    "i am free now":                    "आइ एम फ्री नाउ!",
    "i am late":                        "आइ एम लेट!",
    "i am on my way":                   "आइ एम अन माइ वे!",
    "i am coming":                      "आइ एम कमिङ!",
    "i am here":                        "आइ एम हिअर!",
    "i am ready":                       "आइ एम रेडी!",
    "i am not ready":                   "आइ एम नट रेडी!",
    "i am hungry":                      "आइ एम हङ्ग्री!",
    "i am thirsty":                     "आइ एम थर्स्टी!",
    "i am happy":                       "आइ एम ह्याप्पी!",
    "i am sad":                         "आइ एम स्याड!",
    "i am stressed":                    "आइ एम स्ट्रेस्ड!",
    "i am overwhelmed":                 "आइ एम ओभरवेल्म्ड!",
    "i am frustrated":                  "आइ एम फ्रस्ट्रेटेड!",
    "i am anxious":                     "आइ एम एङ्जियस!",
    "i am excited":                     "आइ एम एक्साइटेड!",
    "i am nervous":                     "आइ एम नर्भस!",
    "i am confused":                    "आइ एम कन्फ्युज्ड!",
    "i am not sure":                    "आइ एम नट स्योर!",
    "i do not know":                    "आइ डु नट नो!",
    "i don't know":                     "आइ डोन्ट नो!",
    "i understand":                     "आइ अन्डरस्ट्यान्ड!",
    "i do not understand":              "आइ डु नट अन्डरस्ट्यान्ड!",
    "i don't understand":               "आइ डोन्ट अन्डरस्ट्यान्ड!",
    "i agree":                          "आइ अग्री!",
    "i disagree":                       "आइ डिसअग्री!",
    "i think so":                       "आइ थिङ्क सो!",
    "i hope so":                        "आइ होप सो!",
    "i need help":                      "आइ नीड हेल्प!",
    "i need more time":                 "आइ नीड मोर टाइम!",
    "i will try":                       "आइ विल ट्राई!",
    "i will be there":                  "आइ विल बी देअर!",
    "i will do it":                     "आइ विल डु इट!",
    "i got it":                         "आइ गट इट!",
    "i see":                            "आइ सी!",
    "i see what you mean":              "आइ सी व्हाट यू मिन!",
    "make sense":                       "मेक्स सेन्स!",
    "that makes sense":                 "द्याट मेक्स सेन्स!",

    # Work phrases (English)
    "the meeting is cancelled":         "द मिटिङ इज क्यान्सेल्ड!",
    "the meeting is at ten":            "द मिटिङ इज एट टेन!",
    "i have a meeting":                 "आइ ह्याभ अ मिटिङ!",
    "i have a meeting in ten minutes":  "आइ ह्याभ अ मिटिङ इन टेन मिनेट्स!",
    "the deadline is today":            "द डेडलाइन इज टुडे!",
    "the deadline is tomorrow":         "द डेडलाइन इज टुमरो!",
    "can you send me the file":         "क्यान यू सेन्ड मी द फाइल?",
    "please send me the report":        "प्लिज सेन्ड मी द रिपोर्ट!",
    "let me know when you are free":    "लेट मी नो व्हेन यू आर फ्री!",
    "i will call you later":            "आइ विल कल यू लेटर!",
    "i will message you":               "आइ विल मेसेज यू!",
    "good job":                         "गुड जब!",
    "good job well done":               "गुड जब! वेल डन!",
    "well done":                        "वेल डन!",
    "great work":                       "ग्रेट वर्क!",
    "keep it up":                       "किप इट अप!",
    "the server is down":               "द सर्भर इज डाउन!",
    "the build failed":                 "द बिल्ड फेल्ड!",
    "the test passed":                  "द टेस्ट पास्ड!",
    "please review my code":            "प्लिज रिभ्यू माइ कोड!",
    "i found a bug":                    "आइ फाउन्ड अ बग!",
    "the bug is fixed":                 "द बग इज फिक्स्ड!",
    "i will fix it":                    "आइ विल फिक्स इट!",
    "it is working now":                "इट इज वर्किङ नाउ!",
    "it is not working":                "इट इज नट वर्किङ!",
    "please update the document":       "प्लिज अपडेट द डकुमेन्ट!",
    "the project is complete":          "द प्रोजेक्ट इज कम्प्लिट!",

    # Farewells
    "see you tomorrow":                 "सी यू टुमरो!",
    "see you later":                    "सी यू लेटर!",
    "see you soon":                     "सी यू सुन!",
    "talk to you later":                "टक टु यू लेटर!",
    "take care":                        "टेक केअर!",
    "take care of yourself":            "टेक केअर अफ योरसेल्फ!",
    "have a good day":                  "ह्याभ अ गुड डे!",
    "have a great day":                 "ह्याभ अ ग्रेट डे!",
    "have a nice day":                  "ह्याभ अ नाइस डे!",
    "have fun":                         "ह्याभ फन!",
    "goodbye":                          "गुडबाई!",
    "bye bye":                          "बाई बाई!",
    "bye":                              "बाई!",

    # Politeness
    "thank you":                        "थ्याङ्क यू!",
    "thank you so much":                "थ्याङ्क यू सो मच!",
    "thank you very much":              "थ्याङ्क यू भेरी मच!",
    "thanks":                           "थ्याङ्क्स!",
    "thanks a lot":                     "थ्याङ्क्स अ लट!",
    "no problem":                       "नो प्रब्लेम!",
    "no worries":                       "नो वरिज!",
    "you are welcome":                  "यू आर वेलकम!",
    "my pleasure":                      "माइ प्लेजर!",
    "sorry":                            "सरी!",
    "i am sorry":                       "आइ एम सरी!",
    "i am so sorry":                    "आइ एम सो सरी!",
    "excuse me":                        "एक्सक्युज मी!",
    "pardon me":                        "पार्डन मी!",
    "please":                           "प्लिज!",
    "of course":                        "अफ कोर्स!",
    "sure":                             "स्योर!",
    "absolutely":                       "एब्सलुटली!",
    "definitely":                       "डेफिनेटली!",

    # Reactions
    "oh my god":                        "ओह माइ गड!",
    "oh no":                            "ओह नो!",
    "oh wow":                           "ओह वाउ!",
    "wow that is amazing":              "वाउ! द्याट इज अमेजिङ!",
    "that is great":                    "द्याट इज ग्रेट!",
    "that is awesome":                  "द्याट इज असम!",
    "that is perfect":                  "द्याट इज पर्फेक्ट!",
    "that is terrible":                 "द्याट इज टेरिबल!",
    "that is funny":                    "द्याट इज फनी!",
    "seriously":                        "सिरियसली?",
    "are you serious":                  "आर यू सिरियस?",
    "no way":                           "नो वे!",
    "that is crazy":                    "द्याट इज क्रेजी!",
    "i can not believe it":             "आइ क्यान नट बिलिभ इट!",
    "i can't believe it":               "आइ क्यान्ट बिलिभ इट!",
    "well done":                        "वेल डन!",
    "good luck":                        "गुड लक!",
    "best of luck":                     "बेस्ट अफ लक!",

    # Daily life
    "how was your day":                 "हाउ वज योर डे?",
    "how was your weekend":             "हाउ वज योर विकेन्ड?",
    "what are your plans":              "व्हाट आर योर प्लान्स?",
    "what are you up to":               "व्हाट आर यू अप टु?",
    "i am going home":                  "आइ एम गोइङ होम!",
    "i am at home":                     "आइ एम एट होम!",
    "i am at the office":               "आइ एम एट द अफिस!",
    "i just woke up":                   "आइ जस्ट वोक अप!",
    "i am having lunch":                "आइ एम ह्याभिङ लन्च!",
    "i just had coffee":                "आइ जस्ट ह्याड कफी!",
    "let us go":                        "लेट अस गो!",
    "let us eat":                       "लेट अस इट!",
    "let us meet":                      "लेट अस मिट!",
    "wait a moment":                    "वेट अ मोमेन्ट!",
    "just a minute":                    "जस्ट अ मिनेट!",
    "one second":                       "वन सेकेन्ड!",
    "hold on":                          "होल्ड अन!",
    "i will be right back":             "आइ विल बी राइट ब्याक!",
    "i am back":                        "आइ एम ब्याक!",

    # ════════════════════════════════════════
    # NEPALI ROMANIZED → Devanagari
    # ════════════════════════════════════════

    # Greetings
    "hey timi k gardai xau":            "हे, तिमी के गर्दैछौ?",
    "hey timi k gardai xu":             "हे, तिमी के गर्दैछु?",
    "timi k gardai xau":                "तिमी के गर्दैछौ?",
    "timi k gardai xu":                 "तिमी के गर्दैछु?",
    "timi k gardai xa":                 "तिमी के गर्दैछ?",
    "timi k xa":                        "तिमी के छ?",
    "timi k xau":                       "तिमी के छौ?",
    "timi kasto xu":                    "तिमी कस्तो छौ?",
    "timi kasto xau":                   "तिमी कस्तो छौ?",
    "tapai kasto hunuhunxa":            "तपाई कस्तो हुनुहुन्छ?",
    "tapai kaha hununxa":               "तपाई कहाँ हुनुहुन्छ?",
    "tapai kaha xa":                    "तपाई कहाँ छ?",
    "namaste tapai lai kasto xa":       "नमस्ते, तपाईलाई कस्तो छ?",
    "ma thik xu":                       "म ठिक छु",
    "ma thik xau":                      "म ठिकै छौ",
    "ma ramro xu":                      "म राम्रो छु",
    "sab thikai xa":                    "सब ठिकै छ",
    "sab thikai xu":                    "सब ठिकै छु",
    "thikai xa tension nalos":          "ठिकै छ, टेन्सन नलोस्",

    # Questions
    "aaja k bhayo":                     "आज के भयो?",
    "k bhayo":                          "के भयो?",
    "k xa":                             "के छ?",
    "k xu":                             "के छु?",
    "k xau":                            "के छौ?",
    "k xaina":                          "के छैन?",
    "k garxau":                         "के गर्छौ?",
    "k garxa":                          "के गर्छ?",
    "k garxu":                          "के गर्छु?",
    "k garne":                          "के गर्ने?",
    "k garnuparxa":                     "के गर्नुपर्छ?",
    "k sochiraxa":                      "के सोचिरहेछ?",
    "timi k sochiraxa":                 "तिमी के सोचिरहेछ?",
    "kina gareko":                      "किन गरेको?",
    "kasari garne":                     "कसरी गर्ने?",
    "kun time ma":                      "कुन समयमा?",

    # Location
    "timi kaha xau":                    "तिमी कहाँ छौ?",
    "timi kaha xu":                     "तिमी कहाँ छु?",
    "tapai kaha xu":                    "तपाई कहाँ छु?",
    "kaha xa":                          "कहाँ छ?",
    "kaha xau":                         "कहाँ छौ?",
    "kaha xu":                          "कहाँ छु?",
    "kaha xaina":                       "कहाँ छैन?",
    "tera saathi kaha xu":              "तेरा साथी कहाँ छु?",

    # Daily Nepali
    "malai thaha xaina":                "मलाई थाहा छैन",
    "malai thaha xa":                   "मलाई थाहा छ",
    "hami sanga time xaina":            "हामीसँग समय छैन",
    "malai bhok lagyo":                 "मलाई भोक लाग्यो",
    "malai tirkha lagyo":               "मलाई तिर्खा लाग्यो",
    "malai ghar janu xa":               "मलाई घर जानु छ",
    "maile bujhina":                    "मैले बुझिन",
    "maile bujhe":                      "मैले बुझे",
    "feri bhannus na":                  "फेरि भन्नुस् न",
    "bistaarai bola":                   "बिस्तारै बोल",
    "ma aaudai xu":                     "म आउँदैछु",
    "ek chin parkhanu":                 "एक छिन पर्खनु",
    "ma busy xu aile":                  "म बिजी छु अहिले",
    "ma free xu aile":                  "म फ्री छु अहिले",

    # Work Nepali/Nepglish
    "ma office jaan lageko xu":         "म अफिस जान लागेको छु",
    "meeting cancel bhayo":             "मिटिङ क्यान्सेल भयो",
    "meeting productive bhayo":         "मिटिङ प्रोडक्टिभ भयो",
    "yo project complete bhayo":        "यो प्रोजेक्ट कम्प्लिट भयो",
    "yo task urgent xu":                "यो टास्क अर्जेन्ट छ",
    "priority high xu":                 "प्रायोरिटी हाइ छ",
    "yo deadline aaj xu":               "यो डेडलाइन आज छ",
    "deadline miss hune bhayo":         "डेडलाइन मिस हुने भयो",
    "target achieve gareko":            "टार्गेट अचिभ गरेको",
    "aba kaam suru garaam":             "अब काम सुरु गरौं",
    "team sanga kura garos":            "टिमसँग कुरा गरोस्",
    "yo task assign gara":              "यो टास्क असाइन गर",
    "client happy xu":                  "क्लाइन्ट ह्याप्पी छ",
    "presentation ready xu":            "प्रेजेन्टेशन रेडी छ",
    "presentation successful bhayo":    "प्रेजेन्टेशन सक्सेसफुल भयो",
    "yo demo ramro bhayo":              "यो डेमो राम्रो भयो",
    "maile report submit gareko":       "मैले रिपोर्ट सबमिट गरेको",
    "maile email gareko":               "मैले इमेल गरेको",
    "yo file send gareko":              "यो फाइल सेन्ड गरेको",
    "maile check gareko":               "मैले चेक गरेको",
    "tapaiko kaam sakiyo":              "तपाईको काम सकियो?",
    "maile sab kaam sakkaye":           "मैले सब काम सकाए",
    "finally done bhayo":               "फाइनली डन भयो",
    "plan change bhayo":                "प्लान चेन्ज भयो",
    "yo idea ramro xa":                 "यो आइडिया राम्रो छ",

    # Feelings Nepali
    "ma dherai exhausted xu":           "म धेरै एग्जस्टेड छु",
    "ma dherai tired xu":               "म धेरै थाकेको छु",
    "ma thakyo dherai":                 "म धेरै थाकेँ",
    "dherai stressed xu":               "धेरै स्ट्रेस्ड छु",
    "dherai pressure xu":               "धेरै प्रेशर छ",
    "ma overwhelmed xu":                "म ओभरवेल्म्ड छु",
    "ma frustrated xu aile":            "म फ्रस्ट्रेटेड छु अहिले",
    "ma anxious xu":                    "म एङ्जियस छु",
    "ma dherai khushi xu":              "म धेरै खुशी छु",
    "ma late hune bhayo":               "म ढिलो हुने भयो",
    "dherai kaam baaki xu":             "धेरै काम बाँकी छ",
    "dherai time lagyo":                "धेरै समय लाग्यो",
    "yo week hectic thiyo":             "यो हप्ता हेक्टिक थियो",
    "next week better hola":            "अर्को हप्ता बेटर होला",

    # Tech Nepali
    "server down xu":                   "सर्भर डाउन छ",
    "database crash bhayo":             "डेटाबेस क्र्यास भयो",
    "build failed bhayo":               "बिल्ड फेल्ड भयो",
    "merge conflict xu":                "मर्ज कन्फ्लिक्ट छ",
    "yo bug fix gareko":                "यो बग फिक्स गरेको",
    "test pass bhayena":                "टेस्ट पास भएन",
    "deploy gareko":                    "डिप्लोय गरेको",
    "pipeline broken xu":               "पाइपलाइन ब्रोकन छ",
    "yo API kaam garena":               "यो API काम गरेन",
    "cache clear garo":                 "क्यास क्लियर गरो",
    "yo service restart gara":          "यो सर्भिस रिस्टार्ट गर",
    "backup liyeko":                    "ब्याकअप लिएको",
    "yo system slow xu":                "यो सिस्टम स्लो छ",
    "yo issue resolve bhayo":           "यो इश्यु रिजल्भ भयो",
    "yo PR review gara":                "यो PR रिभ्यू गर",
    "debug gara ramrari":               "डिबग गर राम्ररी",
    "performance ramro xu":             "परफर्मेन्स राम्रो छ",
    "logic galat xu":                   "लजिक गलत छ",
    "yo feature add gara":              "यो फिचर एड गर",
    "yo version update gara":           "यो भर्सन अपडेट गर",
    "yo function kaam garena":          "यो फन्क्शन काम गरेन",
    "kaam ramro bhayena":               "काम राम्रो भएन",

    # Food/Daily Nepali
    "khaana khayau":                    "खाना खायौ?",
    "maile coffee khaye":               "मैले कफी खाए",
    "ma ghar jaan ready xu":            "म घर जान रेडी छु",
    "saanjh ma bhetaula":               "साँझमा भेटौला",
    "mero phone lost bhayo":            "मेरो फोन हराएको भयो",
    "yo bahira jaaun":                  "बाहिर जाउँ",
    "aba meeting xu":                   "अब मिटिङ छ",

    # ════════════════════════════════════════
    # NEPGLISH MIXED → Devanagari
    # ════════════════════════════════════════

    "aja ma dherai tired xu":               "आज म धेरै थाकेको छु",
    "mero meeting cancel bhayo":            "मेरो मिटिङ क्यान्सेल भयो",
    "yo project complete bhayo client happy xa": "यो प्रोजेक्ट कम्प्लिट भयो, क्लाइन्ट ह्याप्पी छ",
    "ma office ma busy xu":                 "म अफिसमा बिजी छु",
    "deadline aaj xu ma overwhelmed xu":    "डेडलाइन आज छ, म ओभरवेल्म्ड छु",
    "maile coffee khaye meeting ma":        "मैले कफी खाए मिटिङमा",
    "yo bug fix gareko server down thiyo":  "यो बग फिक्स गरेको, सर्भर डाउन थियो",
    "dherai stressed xu aile":              "धेरै स्ट्रेस्ड छु अहिले",
    "team sanga kura gareko feedback ramro thiyo": "टिमसँग कुरा गरेको, फिडब्याक राम्रो थियो",
    "finally done bhayo great feeling xa":  "फाइनली डन भयो, ग्रेट फिलिङ छ",
    "ma happy xu aaja":                     "म ह्याप्पी छु आज",
    "mero naam kashish ho":                 "मेरो नाम कशिश हो",
    "yo kaam difficult xa":                 "यो काम डिफिकल्ट छ",
    "ma nervous xu presentation ko lagi":   "म नर्भस छु प्रेजेन्टेशनको लागि",
    "client call xa aile":                  "क्लाइन्ट कल छ अहिले",
    "report ready bhayena still":           "रिपोर्ट रेडी भएन अझै",
    "yo sprint complete bhayo":             "यो स्प्रिन्ट कम्प्लिट भयो",
    "ma excited xu new project ko lagi":    "म एक्साइटेड छु नयाँ प्रोजेक्टको लागि",
}

# ── WORD MAP ──
WORD_MAP = {

    # ── Pronouns ──
    "ma":           "म",
    "mero":         "मेरो",
    "meri":         "मेरी",
    "maile":        "मैले",
    "malai":        "मलाई",
    "hami":         "हामी",
    "hamro":        "हाम्रो",
    "hamile":       "हामीले",
    "hamilai":      "हामीलाई",
    "timi":         "तिमी",
    "timro":        "तिम्रो",
    "timilai":      "तिमीलाई",
    "timile":       "तिमीले",
    "tapai":        "तपाई",
    "tapaiko":      "तपाईको",
    "tapailai":     "तपाईलाई",
    "tapaile":      "तपाईले",
    "tera":         "तेरा",
    "tero":         "तेरो",
    "u":            "उ",
    "usko":         "उसको",
    "uslai":        "उसलाई",
    "usle":         "उसले",
    "uni":          "उनी",
    "unko":         "उनको",
    "unilai":       "उनीलाई",
    "unile":        "उनीले",
    "uniharu":      "उनीहरू",
    "timiharu":     "तिमीहरू",
    "yo":           "यो",
    "tyo":          "त्यो",
    "yaha":         "यहाँ",
    "tyaha":        "त्यहाँ",
    "aafno":        "आफ्नो",
    "afno":         "आफ्नो",

    # ── Particles / Conjunctions ──
    "ko":           "को",
    "ka":           "का",
    "ki":           "की",
    "ra":           "र",
    "ta":           "त",
    "tara":         "तर",
    "ani":          "अनि",
    "ni":           "नि",
    "nai":          "नै",
    "la":           "ल",
    "lau":          "लौ",
    "hai":          "है",
    "na":           "न",
    "pani":         "पनि",
    "chai":         "चाहिँ",
    "bhane":        "भने",
    "bhanda":       "भन्दा",
    "samma":        "सम्म",
    "dekhi":        "देखि",
    "tira":         "तिर",
    "sanga":        "सँग",
    "saatha":       "साथ",
    "lagi":         "लागि",
    "baahek":       "बाहेक",
    "bahek":        "बाहेक",
    "ma":           "म",     # also "ma" = "मा" (in) handled by context

    # ── Copula / Auxiliary ──
    "xa":           "छ",
    "xu":           "छु",
    "xau":          "छौ",
    "xan":          "छन्",
    "xaina":        "छैन",
    "xainau":       "छैनौ",
    "cha":          "छ",
    "chhu":         "छु",
    "chha":         "छ",
    "chhau":        "छौ",
    "chhan":        "छन्",
    "chaina":       "छैन",
    "ho":           "हो",
    "hoina":        "होइन",
    "hola":         "होला",
    "hos":          "होस्",
    "hunxa":        "हुन्छ",
    "hunxu":        "हुन्छु",
    "hunxau":       "हुन्छौ",
    "hundaina":     "हुँदैन",
    "hudaina":      "हुँदैन",
    "thiyo":        "थियो",
    "thyo":         "थियो",
    "thiyena":      "थिएन",
    "rahecha":      "रहेछ",
    "rahexu":       "रहेछु",

    # ── गर्नु ──
    "garna":        "गर्न",
    "garnu":        "गर्नु",
    "garne":        "गर्ने",
    "garxu":        "गर्छु",
    "garxa":        "गर्छ",
    "garxau":       "गर्छौ",
    "garxan":       "गर्छन्",
    "gardai":       "गर्दै",
    "gardaixu":     "गर्दैछु",
    "gardaixau":    "गर्दैछौ",
    "gardaixa":     "गर्दैछ",
    "gareko":       "गरेको",
    "garena":       "गरेन",
    "garos":        "गरोस्",
    "garaam":       "गरौं",
    "garam":        "गरौं",
    "gara":         "गर",
    "garyo":        "गर्यो",
    "garnuhos":     "गर्नुहोस्",
    "garnuparxa":   "गर्नुपर्छ",
    "garnuparyo":   "गर्नुपर्यो",
    "garnuparne":   "गर्नुपर्ने",
    "garnuparla":   "गर्नुपर्ला",

    # ── हुनु ──
    "hun":          "हुन",
    "hunu":         "हुनु",
    "bhayo":        "भयो",
    "bhayena":      "भएन",
    "bhae":         "भए",
    "bhaye":        "भए",
    "bhaeko":       "भएको",
    "bhayeko":      "भएको",
    "bhaena":       "भएन",

    # ── जानु ──
    "jaan":         "जान",
    "jaanu":        "जानु",
    "janxa":        "जान्छ",
    "janchhu":      "जान्छु",
    "gayeko":       "गएको",
    "gayo":         "गयो",
    "jaau":         "जाउ",
    "jaane":        "जाने",
    "jaaun":        "जाउँ",
    "jaam":         "जाम",
    "jam":          "जाम",
    "jandaixu":     "जाँदैछु",
    "jandaixau":    "जाँदैछौ",

    # ── आउनु ──
    "aau":          "आउ",
    "aaunus":       "आउनुस्",
    "aunxa":        "आउँछ",
    "aunchhu":      "आउँछु",
    "aayeko":       "आएको",
    "aayo":         "आयो",
    "aaudai":       "आउँदै",
    "aaudaixu":     "आउँदैछु",
    "aaudaixau":    "आउँदैछौ",
    "aune":         "आउने",
    "aaun":         "आउँ",

    # ── खानु ──
    "khanu":        "खानु",
    "khanxa":       "खान्छ",
    "khanchhu":     "खान्छु",
    "khaana":       "खाना",
    "khana":        "खाना",
    "khayeko":      "खाएको",
    "khayo":        "खायो",
    "khayau":       "खायौ",
    "khaye":        "खाए",
    "khaau":        "खाउ",
    "khaane":       "खाने",

    # ── लिनु ──
    "linu":         "लिनु",
    "linxa":        "लिन्छ",
    "liyeko":       "लिएको",
    "liyo":         "लियो",
    "lina":         "लिन",

    # ── दिनु ──
    "dinu":         "दिनु",
    "dinxa":        "दिन्छ",
    "diyeko":       "दिएको",
    "diyo":         "दियो",
    "deu":          "देउ",
    "deos":         "देओस्",
    "dina":         "दिन",

    # ── भेट्नु ──
    "bhetaula":     "भेटौला",
    "bhetne":       "भेट्ने",
    "bhetyo":       "भेट्यो",
    "bhetera":      "भेटेर",
    "bheteko":      "भेटेको",

    # ── बस्नु ──
    "basnu":        "बस्नु",
    "basxa":        "बस्छ",
    "baseko":       "बसेको",
    "basa":         "बस",
    "basiraxu":     "बसिरहेछु",
    "basiraxau":    "बसिरहेछौ",
    "basiraheko":   "बसिरहेको",

    # ── हेर्नु ──
    "hernu":        "हेर्नु",
    "herxa":        "हेर्छ",
    "hereko":       "हेरेको",
    "hera":         "हेर",
    "heraula":      "हेरौला",

    # ── सुन्नु ──
    "sunnu":        "सुन्नु",
    "sunxa":        "सुन्छ",
    "suneko":       "सुनेको",
    "suna":         "सुन",

    # ── बोल्नु ──
    "bolnu":        "बोल्नु",
    "bolxa":        "बोल्छ",
    "boleko":       "बोलेको",
    "bola":         "बोल",

    # ── बुझ्नु ──
    "bujhnu":       "बुझ्नु",
    "bujhxa":       "बुझ्छ",
    "bujheko":      "बुझेको",
    "bujhina":      "बुझिन",
    "bujha":        "बुझ",

    # ── सोध्नु ──
    "sodhnu":       "सोध्नु",
    "sodhxa":       "सोध्छ",
    "sodheko":      "सोधेको",
    "sodha":        "सोध",

    # ── भन्नु ──
    "bhan":         "भन",
    "bhannu":       "भन्नु",
    "bhanxa":       "भन्छ",
    "bhaneko":      "भनेको",
    "bhannus":      "भन्नुस्",

    # ── सोच्नु ──
    "sochnu":       "सोच्नु",
    "sochxa":       "सोच्छ",
    "socheko":      "सोचेको",
    "socha":        "सोच",
    "sochiraxu":    "सोचिरहेछु",
    "sochiraxa":    "सोचिरहेछ",

    # ── सक्नु ──
    "saknu":        "सक्नु",
    "sakxa":        "सक्छ",
    "sakxu":        "सक्छु",
    "sakyo":        "सक्यो",
    "sakiyo":       "सकियो",
    "sakkaye":      "सकाए",
    "sakdaina":     "सक्दैन",
    "sakeko":       "सकेको",
    "sakina":       "सकिन",

    # ── लाग्नु ──
    "lagnu":        "लाग्नु",
    "lagxa":        "लाग्छ",
    "lagyo":        "लाग्यो",
    "lagena":       "लागेन",

    # ── मिल्नु ──
    "milnu":        "मिल्नु",
    "milxa":        "मिल्छ",
    "milxu":        "मिल्छु",
    "milyo":        "मिल्यो",
    "mildaina":     "मिल्दैन",

    # ── पर्नु ──
    "parnu":        "पर्नु",
    "parxa":        "पर्छ",
    "paryo":        "पर्यो",
    "parla":        "पर्ला",
    "parne":        "पर्ने",
    "pardaina":     "पर्दैन",

    # ── थाक्नु ──
    "thaknu":       "थाक्नु",
    "thakxa":       "थाक्छ",
    "thakyo":       "थाक्यो",
    "thakeko":      "थाकेको",

    # ── राख्नु ──
    "raakhnu":      "राख्नु",
    "raakhxa":      "राख्छ",
    "raakheko":     "राखेको",
    "raakha":       "राख",
    "raakhos":      "राखोस्",

    # ── पर्खनु ──
    "parkhanu":     "पर्खनु",
    "parkha":       "पर्ख",
    "parkhera":     "पर्खेर",

    # ── Time ──
    "aja":          "आज",
    "aaja":         "आज",
    "hijo":         "हिजो",
    "bholi":        "भोलि",
    "parsi":        "पर्सि",
    "aile":         "अहिले",
    "ahile":        "अहिले",
    "pachhi":       "पछि",
    "pachi":        "पछि",
    "aghi":         "अघि",
    "pahile":       "पहिले",
    "pehile":       "पहिले",
    "aba":          "अब",
    "abba":         "अब",
    "bihaan":       "बिहान",
    "saanjh":       "साँझ",
    "raat":         "रात",
    "din":          "दिन",
    "ghanta":       "घण्टा",
    "minet":        "मिनेट",
    "baela":        "बेला",
    "bela":         "बेला",
    "jaba":         "जब",
    "taba":         "तब",
    "sabera":       "सबेरै",
    "feri":         "फेरि",
    "pheri":        "फेरि",
    "arko":         "अर्को",

    # ── Question words ──
    "k":            "के",
    "ke":           "के",
    "kehi":         "केही",
    "kei":          "केही",
    "kaha":         "कहाँ",
    "kasari":       "कसरी",
    "kata":         "कता",
    "kati":         "कति",
    "kun":          "कुन",
    "kasle":        "कसले",
    "kina":         "किन",
    "kahile":       "कहिले",
    "kaile":        "कहिले",
    "kasko":        "कसको",

    # ── Adverbs ──
    "dherai":       "धेरै",
    "ali":          "अलि",
    "alikai":       "अलिकति",
    "ekdam":        "एकदम",
    "thikai":       "ठिकै",
    "ramrai":       "राम्रै",
    "sahi":         "सही",
    "bilkul":       "बिल्कुल",
    "purai":        "पूरै",
    "jhan":         "झन्",
    "sabai":        "सबै",
    "sab":          "सब",
    "jun":          "जुन",
    "jastai":       "जस्तै",
    "jasto":        "जस्तो",
    "yesto":        "यस्तो",
    "testo":        "त्यस्तो",
    "kohi":         "कोही",
    "kunai":        "कुनै",
    "bistaarai":    "बिस्तारै",
    "bistari":      "बिस्तारै",
    "ramrari":      "राम्ररी",
    "chitto":       "चाँडै",
    "chito":        "चाँडै",
    "still":        "अझै",
    "ajai":         "अझै",
    "ajhai":        "अझै",

    # ── Adjectives ──
    "ramro":        "राम्रो",
    "ramri":        "राम्री",
    "naramro":      "नराम्रो",
    "thulo":        "ठूलो",
    "sano":         "सानो",
    "mitho":        "मिठो",
    "tito":         "तितो",
    "gaaro":        "गाह्रो",
    "sajilo":       "सजिलो",
    "naya":         "नयाँ",
    "puraano":      "पुरानो",
    "purano":       "पुरानो",
    "thik":         "ठिक",
    "galat":        "गलत",
    "sundar":       "सुन्दर",
    "kasto":        "कस्तो",
    "ramailo":      "रमाइलो",

    # ── People ──
    "naam":         "नाम",
    "manche":       "मान्छे",
    "saathi":       "साथी",
    "dost":         "दोस्त",
    "dai":          "दाइ",
    "didi":         "दिदी",
    "bhai":         "भाइ",
    "bahini":       "बहिनी",
    "buwa":         "बुवा",
    "ama":          "आमा",
    "babu":         "बाबु",
    "nani":         "नानी",

    # ── Places ──
    "ghar":         "घर",
    "kaam":         "काम",
    "sarak":        "सडक",
    "baato":        "बाटो",
    "thau":         "ठाउँ",
    "kathmandu":    "काठमाडौँ",
    "nepal":        "नेपाल",
    "bahira":       "बाहिर",
    "baahira":      "बाहिर",
    "bhitra":       "भित्र",
    "maathi":       "माथि",
    "tala":         "तल",

    # ── Food ──
    "dal":          "दाल",
    "bhat":         "भात",
    "roti":         "रोटी",
    "pani":         "पानी",
    "dudh":         "दूध",
    "chiya":        "चिया",

    # ── Health ──
    "bhok":         "भोक",
    "bhokh":        "भोक",
    "tirkha":       "तिर्खा",
    "thaha":        "थाहा",

    # ── Numbers ──
    "ek":           "एक",
    "dui":          "दुई",
    "tin":          "तीन",
    "char":         "चार",
    "paanch":       "पाँच",
    "saat":         "सात",
    "aath":         "आठ",
    "nau":          "नौ",
    "das":          "दस",

    # ── Money ──
    "paisa":        "पैसा",
    "rupaiya":      "रुपैयाँ",

    # ── Greetings ──
    "namaste":      "नमस्ते",
    "namaskar":     "नमस्कार",
    "dhanyabad":    "धन्यवाद",
    "dhanyabaad":   "धन्यवाद",
    "maafi":        "माफी",
    "maaph":        "माफ",
    "maph":         "माफ",
    "swagatam":     "स्वागतम्",
    "hey":          "हे",
    "suru":         "सुरु",
    "anta":         "अन्त",
    "bicha":        "बिच",
    "chin":         "छिन",
    "pal":          "पल",
    "tension":      "टेन्सन",
    "khushi":       "खुशी",
    "khusi":        "खुशी",
    "dukha":        "दुःख",
    "maya":         "माया",

    # ── Work/Office (English loanwords → Nepali phonetic) ──
    "office":           "अफिस",
    "meeting":          "मिटिङ",
    "project":          "प्रोजेक्ट",
    "team":             "टिम",
    "client":           "क्लाइन्ट",
    "deadline":         "डेडलाइन",
    "report":           "रिपोर्ट",
    "email":            "इमेल",
    "presentation":     "प्रेजेन्टेशन",
    "feedback":         "फिडब्याक",
    "target":           "टार्गेट",
    "budget":           "बजेट",
    "task":             "टास्क",
    "plan":             "प्लान",
    "idea":             "आइडिया",
    "design":           "डिजाइन",
    "demo":             "डेमो",
    "priority":         "प्रायोरिटी",

    # ── Tech ──
    "server":           "सर्भर",
    "database":         "डेटाबेस",
    "system":           "सिस्टम",
    "software":         "सफ्टवेयर",
    "code":             "कोड",
    "bug":              "बग",
    "error":            "एरर",
    "feature":          "फिचर",
    "update":           "अपडेट",
    "deploy":           "डिप्लोय",
    "build":            "बिल्ड",
    "test":             "टेस्ट",
    "merge":            "मर्ज",
    "pipeline":         "पाइपलाइन",
    "api":              "API",
    "cache":            "क्यास",
    "backup":           "ब्याकअप",
    "rollback":         "रोलब्याक",
    "conflict":         "कन्फ्लिक्ट",
    "debug":            "डिबग",
    "performance":      "परफर्मेन्स",
    "logic":            "लजिक",
    "function":         "फन्क्शन",
    "module":           "मोड्युल",
    "version":          "भर्सन",
    "sprint":           "स्प्रिन्ट",
    "phone":            "फोन",
    "laptop":           "ल्यापटप",
    "computer":         "कम्प्युटर",
    "internet":         "इन्टरनेट",
    "app":              "एप",

    # ── Emotions (English loanwords used in Nepglish) ──
    "tired":            "थाकेको",
    "exhausted":        "एग्जस्टेड",
    "stressed":         "स्ट्रेस्ड",
    "stress":           "स्ट्रेस",
    "busy":             "बिजी",
    "free":             "फ्री",
    "ready":            "रेडी",
    "late":             "ढिलो",
    "happy":            "ह्याप्पी",
    "sad":              "दुखी",
    "frustrated":       "फ्रस्ट्रेटेड",
    "overwhelmed":      "ओभरवेल्म्ड",
    "anxious":          "एङ्जियस",
    "motivated":        "मोटिभेटेड",
    "excited":          "एक्साइटेड",
    "confident":        "कन्फिडेन्ट",
    "nervous":          "नर्भस",
    "focused":          "फोकस्ड",
    "confused":         "कन्फ्युज्ड",
    "difficult":        "डिफिकल्ट",
    "serious":          "सिरियस",

    # ── Common English action words used in Nepglish ──
    "fix":              "फिक्स",
    "check":            "चेक",
    "send":             "सेन्ड",
    "submit":           "सबमिट",
    "assign":           "असाइन",
    "review":           "रिभ्यू",
    "complete":         "कम्प्लिट",
    "cancel":           "क्यान्सेल",
    "start":            "सुरु",
    "stop":             "रोक्नु",
    "done":             "डन",
    "failed":           "फेल भयो",
    "pass":             "पास",
    "miss":             "मिस",
    "lost":             "हरायो",
    "found":            "फेला पर्यो",
    "coffee":           "कफी",
    "lunch":            "लन्च",
    "break":            "ब्रेक",
    "good":             "राम्रो",
    "bad":              "नराम्रो",
    "nice":             "नाइस",
    "great":            "ग्रेट",
    "perfect":          "परफेक्ट",
    "urgent":           "अर्जेन्ट",
    "slow":             "ढिलो",
    "fast":             "छिटो",
    "new":              "नयाँ",
    "next":             "अर्को",
    "last":             "अन्तिम",
    "finally":          "अन्तमा",
    "already":          "पहिले नै",
    "again":            "फेरि",
    "hectic":           "व्यस्त",
    "productive":       "उत्पादक",
    "successful":       "सफल",
    "pending":          "बाँकी",
    "restart":          "रिस्टार्ट",
    "crash":            "क्र्यास",
    "down":             "डाउन",
    "broken":           "बिग्रिएको",
    "resolve":          "समाधान",
    "resolved":         "समाधान भयो",
    "achieve":          "प्राप्त",
    "change":           "परिवर्तन",
    "better":           "राम्रो",
    "okay":             "ठिकै",
    "ok":               "ठिकै",
    "pressure":         "दबाब",
    "feeling":          "भावना",
    "call":             "कल",
    "file":             "फाइल",
}

print("✅ Word & Sentence maps loaded!")
print(f"📊 Phrase map : {len(PHRASE_MAP)} phrases")
print(f"📊 Word map   : {len(WORD_MAP)} words")

✅ Word & Sentence maps loaded!
📊 Phrase map : 353 phrases
📊 Word map   : 517 words


In [5]:
# ============================================
# CELL 5: Conversion Functions with Fuzzy Match
# ============================================
import re

# ── Whisper correction map ──
WHISPER_CORRECTIONS = [
    # xu/xau fixes
    (r'\bchhu\b',       'xu'),
    (r'\bchha\b',       'xa'),
    (r'\bchhau\b',      'xau'),
    (r'\bchau\b',       'xau'),
    (r'\bchhan\b',      'xan'),
    (r'\bchaina\b',     'xaina'),
    (r'\bcho\b',        'xa'),
    # gardai fixes
    (r'\bgardoyso\b',   'gardai xau'),
    (r'\bgardoyo\b',    'gardai xa'),
    (r'\bgardoisau\b',  'gardai xau'),
    (r'\bgardaiso\b',   'gardai xu'),
    (r'\bgardai so\b',  'gardai xu'),
    (r'\bgardai cho\b', 'gardai xa'),
    # shaudini / garde / kee → gardai xau / ke
    (r'\bshaudini\b',   'xau'),
    (r'\bshaudin\b',    'xau'),
    (r'\bgarde\b',      'gardai'),
    (r'\bgarday\b',     'gardai'),
    (r'\bkee\b',        'ke'),
    (r'\bhuncha\b',     'hunxa'),
    (r'\bhunchha\b',    'hunxa'),
    (r'\bgarcha\b',     'garxa'),
    (r'\bgarchha\b',    'garxa'),
    (r'\bgarchu\b',     'garxu'),
    (r'\bmilcha\b',     'milxa'),
    (r'\bmilchha\b',    'milxa'),
]

# ── Fuzzy word correction map ──
# Maps common Whisper mishearing → correct Romanized word
FUZZY_WORD_MAP = {
    # gardai variants
    "garde":        "gardai",
    "garday":       "gardai",
    "gardey":       "gardai",
    "gardae":       "gardai",
    "gardi":        "gardai",
    # xau/xu/xa variants
    "shaudini":     "xau",
    "shaudin":      "xau",
    "shodini":      "xau",
    "choudini":     "xau",
    "shaudi":       "xau",
    "shodi":        "xu",
    "sho":          "xu",
    "cho":          "xa",
    "chho":         "xa",
    # ke variants
    "kee":          "ke",
    "kay":          "ke",
    "kae":          "ke",
    "ki":           "ke",
    # timi variants
    "timee":        "timi",
    "teemi":        "timi",
    "teme":         "timi",
    "time":         "timi",    # very common Whisper error
    # hey/hello variants
    "helo":         "hello",
    "hallo":        "hello",
    "helo":         "hello",
    # bhayo variants
    "bhyayo":       "bhayo",
    "bhyao":        "bhayo",
    "bhaayo":       "bhayo",
    # xaina variants
    "shaina":       "xaina",
    "chaina":       "xaina",
    "shayna":       "xaina",
    # aaja/aja variants
    "aaj":          "aaja",
    "aajha":        "aaja",
    # thik variants
    "theek":        "thik",
    "theik":        "thik",
    "tik":          "thik",
    # malai variants
    "malaai":       "malai",
    "mala":         "malai",
    # dherai variants
    "dheri":        "dherai",
    "dherey":       "dherai",
    "dhrey":        "dherai",
    # kaha variants
    "kaha":         "kaha",
    "kahaa":        "kaha",
    "kahan":        "kaha",
    # thaha variants
    "thaa":         "thaha",
    "thaaha":       "thaha",
    # hunxa variants
    "huncha":       "hunxa",
    "hunchha":      "hunxa",
    # lagyo variants
    "lagyo":        "lagyo",
    "laagyo":       "lagyo",
    # gareko variants
    "garke":        "gareko",
    "gareko":       "gareko",
    # bistaarai variants
    "bistari":      "bistaarai",
    "bistarai":     "bistaarai",
}


def normalize_whisper_output(text):
    """Apply regex corrections for known Whisper errors."""
    result = text.strip()
    for pattern, replacement in WHISPER_CORRECTIONS:
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)
    return result


def fuzzy_correct_word(word):
    """
    If a word is not in WORD_MAP, check FUZZY_WORD_MAP
    and return the corrected version.
    """
    w = word.lower()
    if w in FUZZY_WORD_MAP:
        return FUZZY_WORD_MAP[w]
    return word


def apply_fuzzy_corrections(text):
    """Apply fuzzy corrections word by word."""
    words = text.split()
    corrected = []
    for word in words:
        # Strip punctuation for lookup
        punct = ''
        clean = word
        if clean and clean[-1] in '.,!?;:':
            punct  = clean[-1]
            clean  = clean[:-1]
        fixed = fuzzy_correct_word(clean)
        corrected.append(fixed + punct)
    return ' '.join(corrected)


def find_best_phrase_match(text_lower):
    """Greedy longest-match in PHRASE_MAP."""
    sorted_phrases = sorted(PHRASE_MAP.keys(), key=len, reverse=True)
    for phrase in sorted_phrases:
        if text_lower.startswith(phrase):
            remaining = text_lower[len(phrase):].lstrip()
            return phrase, PHRASE_MAP[phrase], remaining
    return None


def convert_to_devanagari(text):
    """
    Convert Romanized text → Devanagari.
    1. Phrase map (greedy longest match)
    2. Word map
    3. Unknown → keep Roman (no character guessing)
    """
    if not text:
        return ""

    remaining = text.strip().lower()
    output_parts = []

    while remaining:
        # Phrase match
        match = find_best_phrase_match(remaining)
        if match:
            _, devanagari, remaining = match
            output_parts.append(devanagari)
            continue

        # Next word
        word_match = re.match(r'^([^\s]+)', remaining)
        if not word_match:
            break
        raw_word = word_match.group(1)
        remaining = remaining[len(raw_word):].lstrip()

        # Strip punctuation
        punct = ''
        clean = raw_word
        if clean and clean[-1] in '.,!?;:':
            punct = clean[-1]
            clean = clean[:-1]

        if not clean:
            output_parts.append(punct)
            continue

        # Word map
        if clean in WORD_MAP:
            output_parts.append(WORD_MAP[clean] + punct)
        else:
            # Keep Roman — never character-guess
            output_parts.append(raw_word)

    result = ' '.join(output_parts)
    return re.sub(r'\s+', ' ', result).strip()


def full_convert(raw_text):
    """
    Full pipeline:
    1. Normalize regex corrections
    2. Fuzzy word corrections
    3. Devanagari conversion
    """
    if not raw_text:
        return ""
    step1 = normalize_whisper_output(raw_text)
    step2 = apply_fuzzy_corrections(step1)
    step3 = convert_to_devanagari(step2)
    return step3


# ════════════════════════════════════════════
# Tests
# ════════════════════════════════════════════
print("🧪 Tests\n" + "=" * 65)

tests = [
    # The exact failing case from screenshot
    ("hello kee garde shaudini",        "हेलो, के गर्दैछौ?"),
    # Other Whisper error variants
    ("hello ke gardai xau",             "हेलो, के गर्दैछौ?"),
    ("hey timi k gardai xau",           "हे, तिमी के गर्दैछौ?"),
    ("hey time k garde shaudini",       "हे, तिमी के गर्दैछौ?"),
    # Pure English
    ("hi how are you",                  "हाई! हाउ आर यू?"),
    ("good morning how are you",        "गुड मर्निङ! हाउ आर यू?"),
    ("i am tired today",                "आइ एम टायर्ड टुडे!"),
    ("thank you so much",               "थ्याङ्क यू सो मच!"),
    ("see you tomorrow",                "सी यू टुमरो!"),
    ("the meeting is cancelled",        "द मिटिङ इज क्यान्सेल्ड!"),
    ("good night sleep well",           "गुड नाइट! स्लिप वेल!"),
    ("i found a bug",                   "आइ फाउन्ड अ बग!"),
    # Nepali
    ("ma thik xu",                      "म ठिक छु"),
    ("malai thaha xaina",               "मलाई थाहा छैन"),
    ("meeting cancel bhayo",            "मिटिङ क्यान्सेल भयो"),
    ("finally done bhayo",              "फाइनली डन भयो"),
    # Nepglish
    ("aja ma dherai tired xu",          "आज म धेरै थाकेको छु"),
    ("ma office ma busy xu",            "म अफिसमा बिजी छु"),
]

passed = 0
for inp, expected in tests:
    result = full_convert(inp)
    ok = result == expected
    if ok:
        passed += 1
    icon = "✅" if ok else "⚠️ "
    print(f"{icon} IN : {inp}")
    print(f"   OUT: {result}")
    if not ok:
        print(f"   EXP: {expected}")
    print("-" * 65)

print(f"\n📊 {passed}/{len(tests)} passed")

🧪 Tests
⚠️  IN : hello kee garde shaudini
   OUT: hello के गर्दै छौ
   EXP: हेलो, के गर्दैछौ?
-----------------------------------------------------------------
⚠️  IN : hello ke gardai xau
   OUT: hello के गर्दै छौ
   EXP: हेलो, के गर्दैछौ?
-----------------------------------------------------------------
✅ IN : hey timi k gardai xau
   OUT: हे, तिमी के गर्दैछौ?
-----------------------------------------------------------------
✅ IN : hey time k garde shaudini
   OUT: हे, तिमी के गर्दैछौ?
-----------------------------------------------------------------
✅ IN : hi how are you
   OUT: हाई! हाउ आर यू?
-----------------------------------------------------------------
✅ IN : good morning how are you
   OUT: गुड मर्निङ! हाउ आर यू?
-----------------------------------------------------------------
✅ IN : i am tired today
   OUT: आइ एम टायर्ड टुडे!
-----------------------------------------------------------------
✅ IN : thank you so much
   OUT: थ्याङ्क यू सो मच!
--------------------------------

In [6]:
# ============================================
# CELL 6: FastAPI App — correct logic per language
# ============================================
import os, tempfile
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, nest_asyncio

nest_asyncio.apply()

app = FastAPI(title="Global IME Nepglish API", version="3.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def detect_language_type(text):
    """
    Detect whether transcribed text is English, Nepali, or Nepglish.

    English  → romanized = English,  devanagari = phonetic English  (हाउ आर यू?)
    Nepali   → romanized = Nepali,   devanagari = proper Nepali     (तिमीलाई आज कस्तो छ?)
    Nepglish → romanized = mixed,    devanagari = mixed conversion
    """
    NEPALI_MARKERS = {
        'xu', 'xau', 'xa', 'xaina', 'xan',
        'bhayo', 'bhayena', 'gareko', 'garena',
        'lagyo', 'sakyo', 'thiyo', 'thyo',
        'hunxa', 'hunxu', 'gardai', 'gardaixu', 'gardaixau',
        'malai', 'timilai', 'tapailai', 'hamilai',
        'mero', 'timro', 'hamro', 'tapaiko',
        'aile', 'ahile', 'aba', 'abba',
        'dherai', 'ekdam', 'ramro', 'ramri',
        'garna', 'garnu', 'jaan', 'jaanu',
        'kaha', 'kasari', 'kina', 'kahile',
        'namaste', 'namaskar', 'dhanyabad',
        'bistaarai', 'bistari', 'bihaan', 'saanjh',
        'maile', 'hami', 'timi', 'tapai',
        'malai', 'thaha', 'xaina', 'lagyo',
        'bhok', 'tirkha', 'ghar', 'kaam',
        'garaam', 'garos', 'bhetaula', 'sakiyo',
    }

    ENGLISH_ONLY_MARKERS = {
        'am', 'is', 'are', 'was', 'were',
        'the', 'this', 'that', 'what', 'where',
        'when', 'why', 'how', 'who', 'which',
        'do', 'did', 'does', 'have', 'has', 'had',
        'will', 'would', 'can', 'could', 'should',
        'going', 'doing', 'having', 'getting',
        'morning', 'evening', 'afternoon',
        'goodbye', 'welcome', 'please',
        'my', 'your', 'his', 'her', 'their', 'our',
        'i', 'you', 'he', 'she', 'we', 'they', 'it',
    }

    words = set(text.lower().split())
    nepali_count  = len(words & NEPALI_MARKERS)
    english_count = len(words & ENGLISH_ONLY_MARKERS)

    if nepali_count > 0 and english_count > 0:
        return 'nepglish'
    elif nepali_count > 0:
        return 'nepali'
    elif english_count > 0:
        return 'english'
    else:
        return 'unknown'


@app.get("/")
async def root():
    return {
        "message": "Global IME Nepglish API v3.0",
        "logic": {
            "english_speech":  "romanized=English,  devanagari=phonetic English (हाउ आर यू?)",
            "nepali_speech":   "romanized=Nepali,   devanagari=proper Nepali (तिमीलाई आज कस्तो छ?)",
            "nepglish_speech": "romanized=mixed,    devanagari=mixed conversion",
        }
    }

@app.get("/health")
async def health_check():
    return {
        "status":          "healthy",
        "model":           "Whisper Large V3",
        "phrase_map_size": len(PHRASE_MAP),
        "word_map_size":   len(WORD_MAP),
    }

@app.get("/test")
async def test_endpoint():
    samples = [
        # English → phonetic Devanagari
        ("english",  "how are you"),
        ("english",  "good morning how are you"),
        ("english",  "i am tired today"),
        ("english",  "thank you so much"),
        ("english",  "the meeting is cancelled"),
        # Nepali → proper Devanagari
        ("nepali",   "timilai aaja kasto xa"),
        ("nepali",   "hey timi k gardai xau"),
        ("nepali",   "ma thik xu"),
        ("nepali",   "malai thaha xaina"),
        ("nepali",   "meeting cancel bhayo"),
        # Nepglish → mixed
        ("nepglish", "aaja ma dherai tired xu"),
        ("nepglish", "ma office ma busy xu"),
    ]
    results = []
    for lang, text in samples:
        devanagari = full_convert(text)
        results.append({
            "input":       text,
            "lang_type":   lang,
            "devanagari":  devanagari,
        })
    return {"results": results}


@app.post("/convert")
async def convert_speech(audio: UploadFile = File(...)):
    print(f"\n📥 Received: {audio.filename}")
    tmp_path = None
    try:
        ext = os.path.splitext(audio.filename or "")[1] or ".wav"
        with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
            tmp.write(await audio.read())
            tmp_path = tmp.name

        # ── Step 1: Transcribe → Romanized ──
        print("🎙️  Transcribing...")
        segments, info = model.transcribe(
            tmp_path,
            initial_prompt=WHISPER_INITIAL_PROMPT,
            beam_size=5,
            best_of=5,
            temperature=[0.0, 0.2, 0.4],
            word_timestamps=True,
            vad_filter=True,
            vad_parameters=dict(
                min_silence_duration_ms=500,
                threshold=0.5
            ),
        )
        raw_romanized = " ".join(s.text for s in segments).strip()
        romanized     = normalize_whisper_output(raw_romanized)
        romanized     = apply_fuzzy_corrections(romanized)

        # Detect language type
        lang_type = detect_language_type(romanized)

        print(f"   Raw:       {raw_romanized}")
        print(f"   Romanized: {romanized}")
        print(f"   LangType:  {lang_type}")

        # ── Step 2: English translation ──
        # Always get English translation via Whisper translate task
        print("🔤  Translating to English...")
        seg_en, _ = model.transcribe(
            tmp_path,
            task="translate",
            beam_size=5,
            initial_prompt="Translate this speech to natural English.",
        )
        english = " ".join(s.text for s in seg_en).strip()
        print(f"   English: {english}")

        # ── Step 3: Devanagari conversion ──
        # Logic:
        #   English  → phonetic Devanagari  (how are you → हाउ आर यू?)
        #   Nepali   → proper Nepali script  (timi k gardai xau → तिमी के गर्दैछौ?)
        #   Nepglish → mixed conversion
        print("📝  Converting to Devanagari...")
        devanagari = full_convert(romanized)
        print(f"   Devanagari: {devanagari}")

        os.unlink(tmp_path)
        tmp_path = None

        return JSONResponse(content={
            "success":              True,
            "romanized":            romanized,       # Exactly as spoken (Roman)
            "romanized_raw":        raw_romanized,   # Raw Whisper output
            "english":              english,          # English translation
            "devanagari":           devanagari,       # Devanagari script
            "language_type":        lang_type,        # english / nepali / nepglish
            "detected_language":    info.language,
            "language_probability": round(info.language_probability, 3),
        })

    except Exception as e:
        import traceback
        print(f"❌ {e}\n{traceback.format_exc()}")
        if tmp_path and os.path.exists(tmp_path):
            os.unlink(tmp_path)
        raise HTTPException(status_code=500, detail=str(e))

print("✅ FastAPI app v3.0 ready!")
print("\n📋 Logic:")
print("   🇬🇧 English  → romanized: English    | devanagari: हाउ आर यू?")
print("   🇳🇵 Nepali   → romanized: Nepali     | devanagari: तिमीलाई आज कस्तो छ?")
print("   🔀 Nepglish → romanized: mixed      | devanagari: mixed")

✅ FastAPI app v3.0 ready!

📋 Logic:
   🇬🇧 English  → romanized: English    | devanagari: हाउ आर यू?
   🇳🇵 Nepali   → romanized: Nepali     | devanagari: तिमीलाई आज कस्तो छ?
   🔀 Nepglish → romanized: mixed      | devanagari: mixed


In [7]:
# CELL 7: Start Server
import threading, time

threading.Thread(
    target=lambda: uvicorn.run(
        app, host="0.0.0.0", port=8000, log_level="warning"
    ),
    daemon=True
).start()

time.sleep(3)
print("🚀 Server running → http://0.0.0.0:8000")

🚀 Server running → http://0.0.0.0:8000


In [9]:
# ============================================
# CELL 8: ngrok Tunnel + Verify
# ============================================
from pyngrok import ngrok
import requests

# Kill existing tunnels
ngrok.kill()

# Start tunnel
public_url = ngrok.connect(8000)
print(f"🌐 Public URL: {public_url}")
print(f"📡 API Base:   {public_url}/")
print(f"📋 Docs:       {public_url}/docs")

# Quick verification test
time.sleep(2)
try:
    base = str(public_url).rstrip('/')

    # Test health
    r = requests.get(f"{base}/health")
    print(f"\n✅ Health: {r.json()['status']}")

    # Test transliteration
    r = requests.get(f"{base}/test")
    print(f"\n🧪 Transliteration Tests:")
    for item in r.json()['test_results']:
        print(f"   {item['romanized']}")
        print(f"   → {item['devanagari']}")
        print()

except Exception as e:
    print(f"⚠️ Verification error: {e}")

print(f"\n✅ API Ready!")
print(f"🔗 Share this URL with your frontend: {public_url}")

ERROR:pyngrok.process.ngrok:t=2026-04-26T14:45:06+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.